# ProtoEHR — Local Branch HPO


In [0]:

import os, time, copy, json, math
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, brier_score_loss, balanced_accuracy_score
)

CLASS_NAMES   = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]

# ---- shared model/loss hyperparameters (branch-specific overrides layered on top) ----
BASE_CFG = {
    "flags_path":     "catalog.schema.step7_filtered_cohort_with_apos",
    "output_dir":     "./proto_runs",
    "embedding_dim":  768,
    "adapter_hidden": 256,
    "adapter_out":    64,
    "n_classes":      3,

    "prototypes_per_class": {"control": 6, "pe_ghtn": 8, "gdm": 5},

    "lr": 2e-3,
    "weight_decay": 1e-4,
    "batch_size": 512,
    "max_epochs": 200,
    "patience": 10,

    "lambda_clst": 4e-4, "lambda_sep": 4e-4, "lambda_div": 1.0,
    "lambda_cntrst": 300.0, "lambda_covar": 0.5,
    "zero_diagonal_blocks": True,

    "use_jscl_pretrain": False,

    "max_iterations": 5, "iter_patience": 2, "lr_decay": 0.5,
    "random_seed": 42,
    "run_id": "run_default",
}

GLOBAL_CFG = copy.deepcopy(BASE_CFG)
GLOBAL_CFG["embeddings_path"] = "catalog.schema.patient_clmbr_embeddings_new_v5"

LOCAL_CFG = copy.deepcopy(BASE_CFG)
LOCAL_CFG["embeddings_path"] = "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta"
LOCAL_CFG["topk_pool_k"] = 5
LOCAL_CFG["max_train_patients_per_stratum"] = None  # set an int to cap memory, see Section 2

os.makedirs(BASE_CFG["output_dir"], exist_ok=True)


def total_prototypes(cfg):
    return sum(cfg["prototypes_per_class"][c] for c in CLASS_NAMES)


In [0]:

def build_person_split(cfg):
    """Person-level 80/10/10 stratified split on (control, pe_ghtn, gdm)
    combo. This is the SINGLE source of truth for split membership --
    both branches call this with the same cfg['flags_path'] and
    cfg['random_seed'] and get identical (pid_train, pid_val, pid_test)."""
    flags_df = spark.table(cfg["flags_path"]).select(
        "person_id", "date_of_birth", "recorded_episode_start", "inferred_episode_start",
        "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"
    ).toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)

    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = ((flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1)
                              | (flags_df["has_ghtn"] == 1)).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)

    # age at episode_start -- computed directly (date_of_birth vs episode_start),
    # NOT from a condition-code proxy -- used later as the fusion covariate
    for c in ["date_of_birth", "recorded_episode_start", "inferred_episode_start"]:
        flags_df[c] = pd.to_datetime(flags_df[c])
    flags_df["episode_start"] = flags_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    flags_df["age"] = (flags_df["episode_start"] - flags_df["date_of_birth"]).dt.days / 365.25

    combo = (flags_df["y_control"]*4 + flags_df["y_pe_ghtn"]*2 + flags_df["y_gdm"]).astype(int)
    pid_tv, pid_test = train_test_split(
        flags_df["person_id"].values, test_size=0.10, stratify=combo, random_state=cfg["random_seed"]
    )
    combo_tv = combo[flags_df["person_id"].isin(pid_tv)].values
    pid_train, pid_val = train_test_split(
        pid_tv, test_size=0.10/0.90, stratify=combo_tv, random_state=cfg["random_seed"]
    )
    return {"pid_train": set(pid_train), "pid_val": set(pid_val), "pid_test": set(pid_test),
            "flags_df": flags_df}


def load_global_data(cfg, split, device):
    """Flat, single-embedding-per-patient table -> (X, Y, pid, age) per split."""
    df = spark.table(cfg["embeddings_path"]).toPandas()
    df["patient_id"] = df["patient_id"].astype(str)
    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]
    df = df.merge(split["flags_df"][["person_id"] + label_cols + ["age"]],
                  left_on="patient_id", right_on="person_id", how="inner")

    emb_cols = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == cfg["embedding_dim"]

    out = {}
    for name, pid_set in [("train", split["pid_train"]), ("val", split["pid_val"]), ("test", split["pid_test"])]:
        sub = df[df["patient_id"].isin(pid_set)]
        out[f"X_{name}"] = sub[emb_cols].values.astype(np.float32)
        out[f"Y_{name}"] = sub[label_cols].values.astype(np.float32)
        out[f"pid_{name}"] = sub["patient_id"].values
        out[f"age_{name}"] = sub["age"].values.astype(np.float32)

    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    print(f"[global] train={len(out['X_train'])} val={len(out['X_val'])} test={len(out['X_test'])}")
    return out


def load_local_data(cfg, split, device):
    """Long-format rolling-delta table -> row-level (X, Y-broadcast, pid, age)
    per split, dropping NaN-embedding rows (delta not yet computable / source
    window unavailable). Uses the chunked patient-filtered pull pattern
    established earlier in this project so peak memory stays bounded, with
    an optional per-stratum training cap (cfg['max_train_patients_per_stratum']).
    """
    flags_df = split["flags_df"]
    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]

    avail_pids = spark.sql(
        f"SELECT DISTINCT patient_id FROM {cfg['embeddings_path']} WHERE unavailable = 0"
    ).toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)
    patient_labels = avail_pids.merge(flags_df, left_on="patient_id", right_on="person_id", how="inner")

    def pull_for(pid_set, cap_per_stratum=None):
        pids = patient_labels[patient_labels["patient_id"].isin(pid_set)].copy()
        if cap_per_stratum is not None:
            combo = (pids["y_control"]*4 + pids["y_pe_ghtn"]*2 + pids["y_gdm"]).astype(int).values
            rng = np.random.RandomState(cfg["random_seed"])
            keep = []
            for stratum in np.unique(combo):
                idxs = np.where(combo == stratum)[0]
                if len(idxs) > cap_per_stratum:
                    idxs = rng.choice(idxs, size=cap_per_stratum, replace=False)
                keep.extend(idxs.tolist())
            pids = pids.iloc[sorted(keep)]

        chunk_size, chunks = 2000, []
        ids = pids["patient_id"].tolist()
        for i in range(0, len(ids), chunk_size):
            batch_ids = ids[i:i + chunk_size]
            id_list_sql = ", ".join(f"'{pid}'" for pid in batch_ids)
            chunk_df = spark.sql(f'''
                SELECT * FROM {cfg['embeddings_path']}
                WHERE patient_id IN ({id_list_sql}) AND unavailable = 0
            ''').toPandas()
            emb_cols = sorted([c for c in chunk_df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
            chunk_df[emb_cols] = chunk_df[emb_cols].astype(np.float32)
            chunk_df["patient_id"] = chunk_df["patient_id"].astype(str)
            chunks.append(chunk_df)
        rows = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        if rows.empty:
            return rows, []
        emb_cols = sorted([c for c in rows.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
        nan_mask = rows[emb_cols].isna().any(axis=1)
        rows = rows.loc[~nan_mask].reset_index(drop=True)
        rows = rows.merge(pids[["patient_id"] + label_cols + ["age"]], on="patient_id", how="inner")
        return rows, emb_cols

    train_rows, emb_cols = pull_for(split["pid_train"], cfg.get("max_train_patients_per_stratum"))
    val_rows, _ = pull_for(split["pid_val"])
    test_rows, _ = pull_for(split["pid_test"])

    out = {}
    for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
        out[f"X_{name}"] = rows[emb_cols].values.astype(np.float32)
        out[f"Y_{name}"] = rows[label_cols].values.astype(np.float32)
        out[f"pid_{name}"] = rows["patient_id"].values
        out[f"age_{name}"] = rows["age"].values.astype(np.float32)
        out[f"window_name_{name}"] = rows["window_name"].values if "window_name" in rows.columns else np.array([None]*len(rows))

    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    print(f"[local]  train={len(out['X_train'])} rows / {len(set(out['pid_train']))} patients | "
          f"val={len(out['X_val'])} rows | test={len(out['X_test'])} rows")
    return out


In [0]:

def class_offsets(prototypes_per_class):
    offsets, start = {}, 0
    for name in CLASS_NAMES:
        k = prototypes_per_class[name]
        offsets[name] = (start, start + k)
        start += k
    return offsets, start


class AdapterMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)


class PrototypeLayer(nn.Module):
    def __init__(self, prototypes_per_class, prototype_dim, scale, verbose=True):
        super().__init__()
        self.prototypes_per_class = prototypes_per_class
        self.offsets, self.n_prototypes = class_offsets(prototypes_per_class)
        self.prototype_dim = prototype_dim
        self.scale = scale
        self.prototypes = nn.Parameter(torch.randn(self.n_prototypes, prototype_dim) * 0.1)

        assignment = torch.zeros(len(CLASS_NAMES), self.n_prototypes)
        for c, name in enumerate(CLASS_NAMES):
            s, e = self.offsets[name]
            assignment[c, s:e] = 1.0
        self.register_buffer("assignment", assignment)
        if verbose:
            print(f"  prototypes: {self.n_prototypes} total -> " +
                  ", ".join(f"{n}={self.offsets[n][1]-self.offsets[n][0]}" for n in CLASS_NAMES))

    def forward(self, z):
        z_norm = F.normalize(z, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        return self.scale * (z_norm @ p_norm.T)


class ProtoEHRNet(nn.Module):
    def __init__(self, cfg, scale, verbose=True):
        super().__init__()
        self.adapter = AdapterMLP(cfg["embedding_dim"], cfg["adapter_hidden"], cfg["adapter_out"])
        self.proto_layer = PrototypeLayer(cfg["prototypes_per_class"], cfg["adapter_out"], scale, verbose)
        self.classifier = nn.Linear(self.proto_layer.n_prototypes, cfg["n_classes"], bias=True)

    def forward(self, x):
        z = self.adapter(x)
        sim = self.proto_layer(z)
        logits = self.classifier(sim)
        return logits, sim, z


In [0]:

def bce_loss(logits, labels, class_weights):
    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=class_weights, reduction="mean")

def clustering_loss(sim, labels, assignment):
    pos_mask = (labels @ assignment).clamp(0, 1).bool()
    masked = sim.masked_fill(~pos_mask, -1e9)
    has_pos = pos_mask.any(dim=1)
    return -masked[has_pos].max(dim=1).values.mean() if has_pos.any() else sim.new_tensor(0.0)

def separation_loss(sim, labels, assignment):
    neg_mask = ((labels @ assignment) == 0)
    masked = sim.masked_fill(~neg_mask, -1e9)
    has_neg = neg_mask.any(dim=1)
    return masked[has_neg].max(dim=1).values.mean() if has_neg.any() else sim.new_tensor(0.0)

def orthogonality_loss(prototypes):
    p = F.normalize(prototypes, dim=1)
    gram = p @ p.T
    I = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - I, p="fro") ** 2

def build_jaccard_matrix(Y_train, prototypes_per_class, zero_diagonal_blocks=True):
    n_classes = len(CLASS_NAMES)
    C_class = np.zeros((n_classes, n_classes))
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            inter = ((Y_train[:, c1] == 1) & (Y_train[:, c2] == 1)).sum()
            union = ((Y_train[:, c1] == 1) | (Y_train[:, c2] == 1)).sum()
            C_class[c1, c2] = inter / union if union > 0 else 0.0
    if zero_diagonal_blocks:
        np.fill_diagonal(C_class, 0.0)
    offsets, P = class_offsets(prototypes_per_class)
    C_proto = np.zeros((P, P))
    for c1, n1 in enumerate(CLASS_NAMES):
        for c2, n2 in enumerate(CLASS_NAMES):
            rs, re = offsets[n1]; cs, ce = offsets[n2]
            C_proto[rs:re, cs:ce] = C_class[c1, c2]
    return torch.tensor(C_proto, dtype=torch.float32)

def contrastive_loss(prototypes, C):
    p = F.normalize(prototypes, dim=1)
    S = p @ p.T
    C_comp = 1.0 - C
    pos_term = (C * S).sum() / C.sum().clamp(min=1e-8)
    neg_term = (C_comp * S).sum() / C_comp.sum().clamp(min=1e-8)
    return -(1.0 / prototypes.size(0) ** 0.5) * (pos_term - neg_term)

def covariance_loss(z, eps=1e-4):
    z = z - z.mean(dim=0, keepdim=True)
    n, d = z.shape
    cov = (z.T @ z) / (n - 1)
    off_diag = cov - torch.diag(torch.diagonal(cov))
    return (off_diag ** 2).sum() / d


In [0]:

def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    results["auroc_macro"] = macro("auroc")
    results["auprc_macro"] = macro("auprc")
    results["f1_macro"] = macro("f1")
    results["bacc_macro"] = macro("bacc")
    results["brier_macro"] = macro("brier")

    out_auroc = [results[f"auroc_{n}"] for n in outcome_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro_outcomes_only"] = float(np.mean(out_auroc)) if out_auroc else float("nan")
    out_auprc = [results[f"auprc_{n}"] for n in outcome_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro_outcomes_only"] = float(np.mean(out_auprc)) if out_auprc else float("nan")

    results["_primary"] = results["auroc_macro"]
    return results


def evaluate_all(model, X, Y, device, return_probs=False):
    """GLOBAL branch: row-level (one embedding per patient already)."""
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(torch.tensor(X, dtype=torch.float32).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
    results = _compute_metric_suite(Y, probs)
    return (results, probs) if return_probs else results


@torch.no_grad()
def topk_pool_patient_probs(model, X, pid, device, k=5, batch_size=512):
    """LOCAL branch: row-level forward pass, then top-k pooling of
    per-window prototype similarities into one pooled vector per
    patient, classified once per patient."""
    model.eval()
    all_sim = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32).to(device)
        z = model.adapter(xb)
        all_sim.append(model.proto_layer(z).cpu().numpy())
    sim_all = np.concatenate(all_sim, axis=0)
    n_p = sim_all.shape[1]
    proto_cols = [f"p{i}" for i in range(n_p)]
    df = pd.DataFrame(sim_all, columns=proto_cols)
    df["pid"] = np.asarray(pid)

    def pool(g):
        vals = g[proto_cols].values
        k_eff = min(k, vals.shape[0])
        return pd.Series(np.sort(vals, axis=0)[-k_eff:, :].mean(axis=0), index=proto_cols)

    pooled = df.groupby("pid", sort=False).apply(pool)
    pooled_pid = pooled.index.values
    pooled_sim = pooled.values.astype(np.float32)
    logits = model.classifier(torch.tensor(pooled_sim, dtype=torch.float32).to(device))
    return pooled_pid, torch.sigmoid(logits).cpu().numpy(), pooled_sim


def evaluate_patient_level(model, X, Y, pid, device, k=5, return_probs=False):
    pooled_pid, probs, _ = topk_pool_patient_probs(model, X, pid, device, k=k)
    y_df = pd.DataFrame(Y, columns=[f"y{i}" for i in range(Y.shape[1])])
    y_df["pid"] = np.asarray(pid)
    y_patient = y_df.drop_duplicates("pid").set_index("pid").loc[pooled_pid].values.astype(np.float32)
    results = _compute_metric_suite(y_patient, probs)
    return (results, probs, pooled_pid) if return_probs else results


def prototype_diagnostics(model):
    offsets, _ = class_offsets(model.proto_layer.prototypes_per_class)
    with torch.no_grad():
        p = F.normalize(model.proto_layer.prototypes.cpu(), dim=1)
        gram = (p @ p.T).numpy()
    concentrations = []
    for name in CLASS_NAMES:
        s, e = offsets[name]; k = e - s
        if k <= 1:
            continue
        block = gram[s:e, s:e]
        concentrations.append((block.sum() - np.trace(block)) / (k * (k - 1)))
    return {"proto_concentration": float(np.mean(concentrations)) if concentrations else float("nan")}


In [0]:

RISK_FACTOR_CONDITIONS = {
    "pe_ghtn": {
        "chronic_hypertension": [319826, 4322024, 317898, 4118910, 4311246, 443771, 4339214,
                                 320128, 381290, 312902, 4013643, 321080, 321074, 312648],
        "diabetes": [4008576, 4024659, 4058243, 195771, 43531008, 4193704, 43531010, 201820,
                    201826, 43531007, 201254, 443412],
        "kidney_disease": [43531578, 40483287, 46271022, 443611, 443601, 44784621, 43021852,
                           443597, 45763854, 45763855, 443612, 198124, 443614, 44782429,
                           44782690, 192279, 45757356],
        "autoimmune_disease": [257628, 4149913, 4066824, 4344158, 4063581, 4055640, 4343923, 4098292],
        "obesity": [433736, 434005, 4097996, 4100857, 45757112, 42872398],
        "hyperthyroidism": [4337834],
    },
    "gdm": {
        "obesity": [433736, 434005, 4097996, 4100857, 45757112, 42872398],
        "pcos": [36683296, 40443308],
        "prediabetes": [4311629],
    },
}
# maternal_age handled via direct age computation (already have `age` per patient), not condition codes


def _population_baseline(cfg, class_name):
    where = ("has_pe = 1 OR has_inferred_pe = 1 OR has_ghtn = 1" if class_name == "pe_ghtn"
             else "has_gdm = 1")
    pids_df = spark.sql(f"SELECT person_id FROM {cfg['flags_path']} WHERE {where}").toPandas()
    patient_ids = pids_df["person_id"].astype(str).tolist()
    if not patient_ids:
        return {}
    id_list_sql = ", ".join(f"'{p}'" for p in patient_ids)
    conds = spark.sql(f'''
        SELECT person_id, condition_concept_id, condition_source_concept_id
        FROM catalog.schema.condition_occurrence_cohort WHERE person_id IN ({id_list_sql})
    ''').toPandas()
    conds["person_id"] = conds["person_id"].astype(str)
    # explicit loop instead of groupby(...).apply(lambda g: set(...)) --
    # returning a raw python set from a groupby-apply lambda has
    # inconsistent behavior across pandas versions (sets are iterable,
    # so pandas sometimes tries to expand them instead of treating each
    # as one aggregated value per group) -- this avoids that ambiguity.
    concept_by_pid = {}
    for pid, g in conds.groupby("person_id"):
        concept_by_pid[pid] = (set(g["condition_concept_id"].dropna().astype(int))
                               | set(g["condition_source_concept_id"].dropna().astype(int)))
    baseline = {}
    for cat, ids in RISK_FACTOR_CONDITIONS[class_name].items():
        ids_set = set(ids)
        hits = sum(1 for pid in patient_ids if bool(concept_by_pid.get(pid, set()) & ids_set))
        baseline[cat] = hits / len(patient_ids)
    return baseline


_BASELINE_CACHE = {}

def compute_risk_factor_ratios(model, X_train, Y_train, pid_train, device, cfg,
                               pooled=False, pid_pool_source=None):
    """For each class, projects each prototype onto its nearest same-class
    training patient (simple argmax, no cross-class/comorbid exclusion here
    -- this is a lightweight per-trial DIAGNOSTIC, not the final projection
    pipeline), then returns {class: {category: ratio}} = prototype hit rate
    / population baseline. Cached population baselines across trials within
    a process (baselines don't change between HPO trials)."""
    model.eval()
    with torch.no_grad():
        all_z = []
        for start in range(0, len(X_train), 512):
            xb = torch.tensor(X_train[start:start+512], dtype=torch.float32).to(device)
            all_z.append(model.adapter(xb))
        z_norm = F.normalize(torch.cat(all_z, dim=0), dim=1)
        p_norm = F.normalize(model.proto_layer.prototypes, dim=1)

    offsets = model.proto_layer.offsets
    id_list_all = list(set(pid_pool_source if pid_pool_source is not None else pid_train))
    id_list_sql = ", ".join(f"'{p}'" for p in id_list_all)
    conds = spark.sql(f'''
        SELECT person_id, condition_concept_id, condition_source_concept_id
        FROM catalog.schema.condition_occurrence_cohort WHERE person_id IN ({id_list_sql})
    ''').toPandas()
    conds["person_id"] = conds["person_id"].astype(str)
    concept_by_pid = {}
    for pid, g in conds.groupby("person_id"):
        concept_by_pid[pid] = (set(g["condition_concept_id"].dropna().astype(int))
                               | set(g["condition_source_concept_id"].dropna().astype(int)))

    results = {}
    for class_name, cats in RISK_FACTOR_CONDITIONS.items():
        if class_name not in offsets:
            continue
        cache_key = (cfg["flags_path"], class_name)
        if cache_key not in _BASELINE_CACHE:
            _BASELINE_CACHE[cache_key] = _population_baseline(cfg, class_name)
        baseline = _BASELINE_CACHE[cache_key]

        s, e = offsets[class_name]
        class_idx = CLASS_NAMES.index(class_name)
        class_mask = Y_train[:, class_idx] == 1
        if not class_mask.any():
            continue
        eligible_z = z_norm[torch.tensor(class_mask, device=device)]
        eligible_pid = np.asarray(pid_train)[class_mask]
        p_block = p_norm[s:e]
        sims = (p_block @ eligible_z.T).cpu().numpy()
        best_idx = sims.argmax(axis=1)
        proto_pids = eligible_pid[best_idx]

        cat_ratios = {}
        for cat, ids in cats.items():
            ids_set = set(ids)
            hits = sum(1 for pid in proto_pids if bool(concept_by_pid.get(pid, set()) & ids_set))
            proto_rate = hits / len(proto_pids)
            base_rate = baseline.get(cat, np.nan)
            cat_ratios[cat] = proto_rate / base_rate if base_rate and base_rate > 0 else float("nan")
        results[class_name] = cat_ratios
    return results


In [0]:

def project_prototypes(model, X_train, Y_train, pid_train, device, verbose=True):
    model.eval()
    with torch.no_grad():
        all_z = []
        for start in range(0, len(X_train), 512):
            xb = torch.tensor(X_train[start:start+512], dtype=torch.float32).to(device)
            all_z.append(model.adapter(xb))
        all_z = torch.cat(all_z, dim=0)
        z_norm = F.normalize(all_z, dim=1)
        p_norm_all = F.normalize(model.proto_layer.prototypes, dim=1)
        scale = model.proto_layer.scale
        n_train = len(X_train)
        used_mask = torch.zeros(n_train, dtype=torch.bool, device=device)
        n_labels_positive = torch.tensor(Y_train.sum(axis=1), device=device)
        purity_mask = (n_labels_positive == 1)

        proj = {}
        for name in CLASS_NAMES:
            s, e = model.proto_layer.offsets[name]
            class_idx = CLASS_NAMES.index(name)
            class_mask = torch.tensor(Y_train[:, class_idx] == 1, device=device)
            if not class_mask.any():
                continue
            p_block = p_norm_all[s:e]
            n_prototypes_this_class = e - s

            for k in range(n_prototypes_this_class):
                p_idx = s + k
                # PREFER pure (single-label) unclaimed patients; fall back to
                # ANY unclaimed same-class patient only if the pure pool is
                # exhausted for this class at this point in the loop.
                pure_eligible = class_mask & (~used_mask) & purity_mask
                eligible_mask = pure_eligible if pure_eligible.any() else (class_mask & (~used_mask))
                if not eligible_mask.any():
                    if verbose:
                        print(f"  WARNING: class '{name}' ran out of eligible patients entirely "
                              f"at prototype slot {k+1}/{n_prototypes_this_class} -- skipping this slot.")
                    continue

                eligible_z = z_norm[eligible_mask]
                eligible_idxs = torch.where(eligible_mask)[0]
                sims = scale * (p_block[k:k+1] @ eligible_z.T).squeeze(0)
                best_local = sims.argmax().item()
                train_idx = int(eligible_idxs[best_local].item())
                best_sim = float(sims[best_local].item())

                model.proto_layer.prototypes[p_idx].copy_(all_z[train_idx])
                proj[p_idx] = {"train_idx": train_idx, "patient_id": str(pid_train[train_idx]),
                              "class_idx": class_idx, "class_name": name,
                              "sim_score": best_sim,
                              "used_comorbid_fallback": not bool(purity_mask[train_idx].item())}
                used_mask[train_idx] = True
    if verbose:
        n_fallback = sum(1 for v in proj.values() if v["used_comorbid_fallback"])
        print(f"  projected {len(proj)} prototypes ({n_fallback} used the comorbid fallback)")
    return proj


def run_one_epoch(model, loader, optimizer, C_jac, class_weights, cfg, device, train=True):
    model.train(train)
    totals = {k: torch.zeros((), device=device) for k in
             ["total", "bce", "clst", "sep", "div", "cntrst", "covar"]}
    n_batches = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_b, Y_b in loader:
            X_b, Y_b = X_b.to(device, non_blocking=True), Y_b.to(device, non_blocking=True)
            logits, sim, z = model(X_b)
            l_bce = bce_loss(logits, Y_b, class_weights)
            l_clst = clustering_loss(sim, Y_b, model.proto_layer.assignment)
            l_sep = separation_loss(sim, Y_b, model.proto_layer.assignment)
            l_div = orthogonality_loss(model.proto_layer.prototypes)
            l_cntrst = contrastive_loss(model.proto_layer.prototypes, C_jac)
            l_covar = covariance_loss(z)
            l_total = (l_bce + cfg["lambda_clst"]*l_clst + cfg["lambda_sep"]*l_sep
                       + cfg["lambda_div"]*l_div + cfg["lambda_cntrst"]*l_cntrst
                       + cfg.get("lambda_covar", 0.0)*l_covar)
            if train:
                optimizer.zero_grad(set_to_none=True)
                l_total.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            totals["total"] += l_total.detach(); totals["bce"] += l_bce.detach()
            totals["clst"] += l_clst.detach(); totals["sep"] += l_sep.detach()
            totals["div"] += l_div.detach(); totals["cntrst"] += l_cntrst.detach()
            totals["covar"] += l_covar.detach()
            n_batches += 1
    return {k: (v / max(n_batches, 1)).item() for k, v in totals.items()}


def train_protoehr(cfg, data, device, eval_fn, verbose=True, branch_name="", trial=None):
    torch.manual_seed(cfg["random_seed"]); np.random.seed(cfg["random_seed"])
    X_train, Y_train, pid_train = data["X_train"], data["Y_train"], data["pid_train"]
    X_val, Y_val, pid_val = data["X_val"], data["Y_val"], data["pid_val"]
    class_weights = data["class_weights"].to(device)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True)

    scale = cfg["adapter_out"] ** 0.5
    model = ProtoEHRNet(cfg, scale, verbose=verbose).to(device)
    C_jac = build_jaccard_matrix(Y_train, cfg["prototypes_per_class"], cfg["zero_diagonal_blocks"]).to(device)

    best_iter_metric, iter_patience_ctr = -1.0, 0
    current_lr = cfg["lr"]
    overall_best_state = None
    projection_map = {}

    for iteration in range(1, cfg["max_iterations"] + 1):
        optimizer = torch.optim.Adam(model.parameters(), lr=current_lr, weight_decay=cfg["weight_decay"])
        best_epoch_metric, epoch_patience, best_epoch_state = -1.0, 0, None

        for epoch in range(1, cfg["max_epochs"] + 1):
            run_one_epoch(model, train_loader, optimizer, C_jac, class_weights, cfg, device, train=True)
            va = eval_fn(model, X_val, Y_val, pid_val, device)
            primary = va["_primary"]
            improved = (not np.isnan(primary)) and (primary > best_epoch_metric)
            if improved or epoch == 1:
                if improved:
                    best_epoch_metric = primary; epoch_patience = 0
                best_epoch_state = {k: v.clone() for k, v in model.state_dict().items()}
                if not improved:
                    epoch_patience += 1
            else:
                epoch_patience += 1
            if epoch_patience >= cfg["patience"]:
                break

        model.load_state_dict(best_epoch_state)
        if best_epoch_metric > best_iter_metric:
            best_iter_metric = best_epoch_metric; iter_patience_ctr = 0
            overall_best_state = copy.deepcopy(best_epoch_state)
        else:
            iter_patience_ctr += 1

        projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
        if verbose:
            print(f"  [{branch_name}] iter {iteration}/{cfg['max_iterations']} "
                  f"best={best_epoch_metric:.4f} all-time={best_iter_metric:.4f} "
                  f"iter_pat={iter_patience_ctr}/{cfg['iter_patience']}")

        if trial is not None:
            trial.report(best_iter_metric, step=iteration)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if iter_patience_ctr >= cfg["iter_patience"]:
            model.load_state_dict(overall_best_state)
            projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
            break
        current_lr *= cfg["lr_decay"]

    model.load_state_dict(overall_best_state)
    final_metrics = eval_fn(model, X_val, Y_val, pid_val, device)
    proto_diag = prototype_diagnostics(model)
    return model, final_metrics, proto_diag, projection_map


In [0]:

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def print_progress_callback(study, trial, every=5):
    if (trial.number + 1) % every == 0:
        print(f"    [{study.study_name}] trial {trial.number + 1} done -- "
              f"best so far: {study.best_value:.4f}")


LR_GRID = [5e-4, 1e-3, 1.5e-3, 2e-3, 2.5e-3, 3e-3]


def make_eval_fn(branch, k_pool=5):
    if branch == "global":
        return lambda model, X, Y, pid, device: evaluate_all(model, X, Y, device)
    else:
        return lambda model, X, Y, pid, device: evaluate_patient_level(model, X, Y, pid, device, k=k_pool)


def sample_phase_a(trial, base_cfg, min_k=1, max_k=20):
    cfg = copy.deepcopy(base_cfg)
    cfg["prototypes_per_class"] = {
        "control": trial.suggest_int("k_control", min_k, max_k),
        "pe_ghtn": trial.suggest_int("k_pe_ghtn", min_k, max_k),
        "gdm":     trial.suggest_int("k_gdm", min_k, max_k),
    }
    cfg["run_id"] = f"phaseA_trial_{trial.number}"
    return cfg


def sample_phase_b(trial, base_cfg):
    cfg = copy.deepcopy(base_cfg)
    cfg["lambda_clst"]   = trial.suggest_float("lambda_clst", 1e-4, 1e-1, log=True)
    cfg["lambda_sep"]    = trial.suggest_float("lambda_sep", 1e-5, 1e-2, log=True)
    cfg["lambda_div"]    = trial.suggest_float("lambda_div", 1e-2, 50.0, log=True)
    cfg["lambda_cntrst"] = trial.suggest_float("lambda_cntrst", 1.0, 500.0, log=True)
    cfg["lambda_covar"]  = trial.suggest_float("lambda_covar", 1e-2, 20.0, log=True)
    cfg["lr"]            = trial.suggest_categorical("lr", LR_GRID)
    cfg["run_id"]        = f"phaseB_trial_{trial.number}"
    return cfg


def make_objective(data, device, branch, sample_fn, base_cfg, hpo_epochs=60, hpo_iterations=3, **sample_kwargs):
    eval_fn = make_eval_fn(branch, k_pool=base_cfg.get("topk_pool_k", 5))

    def objective(trial):
        cfg = sample_fn(trial, base_cfg, **sample_kwargs)
        cfg["max_epochs"], cfg["max_iterations"] = hpo_epochs, hpo_iterations
        cfg["_branch_name"] = branch
        try:
            model, final_metrics, proto_diag, projection_map = train_protoehr(
                cfg, data, device, eval_fn, verbose=False, branch_name=branch, trial=trial
            )
        except optuna.TrialPruned:
            raise
        except Exception as e:
            print(f"[{branch} trial {trial.number}] FAILED: {e}")
            return 0.0

        # OPTIMIZE ON auroc_macro ONLY -- print every other macro metric
        # for visibility, but only auroc_macro drives Optuna's search.
        print(f"[{branch} trial {trial.number}] "
              f"auroc_macro={final_metrics['auroc_macro']:.4f} (OPTIMIZED)  |  "
              f"auprc_macro={final_metrics['auprc_macro']:.4f}  "
              f"f1_macro={final_metrics['f1_macro']:.4f}  "
              f"bacc_macro={final_metrics['bacc_macro']:.4f}  "
              f"brier_macro={final_metrics['brier_macro']:.4f}")

        trial.set_user_attr("final_metrics", final_metrics)
        trial.set_user_attr("proto_diag", proto_diag)
        trial.set_user_attr("cfg", cfg)
        return final_metrics["auroc_macro"]
    return objective


def log_risk_factor_ratios_for_trial(trial, data, device, verbose=True):
    cfg = trial.user_attrs["cfg"]
    eval_fn = make_eval_fn(cfg.get("_branch_name", "global"), k_pool=cfg.get("topk_pool_k", 5))
    model, final_metrics, proto_diag, projection_map = train_protoehr(
        cfg, data, device, eval_fn, verbose=False
    )
    ratios = compute_risk_factor_ratios(
        model, data["X_train"], data["Y_train"], data["pid_train"], device, cfg,
        pid_pool_source=data["pid_train"]
    )
    if verbose:
        print(f"Risk-factor ratios for trial #{trial.number}: {ratios}")
    return ratios


def _enqueue_baseline_phase_a(study, base_cfg):
    """Adds your known-good baseline architecture as an explicit trial,
    evaluated on the SAME reduced HPO budget/metric as every other trial
    -- gives you a true apples-to-apples comparison point, and TPE uses
    it to inform where nearby configs are likely to also do well."""
    study.enqueue_trial({
        "k_control": base_cfg["prototypes_per_class"]["control"],
        "k_pe_ghtn": base_cfg["prototypes_per_class"]["pe_ghtn"],
        "k_gdm":     base_cfg["prototypes_per_class"]["gdm"],
    })


def _enqueue_baseline_phase_b(study, base_cfg):
    """Same idea for Phase B -- your baseline's actual loss weights + lr,
    not a re-sampled guess."""
    study.enqueue_trial({
        "lambda_clst":   base_cfg["lambda_clst"],
        "lambda_sep":    base_cfg["lambda_sep"],
        "lambda_div":    base_cfg["lambda_div"],
        "lambda_cntrst": base_cfg["lambda_cntrst"],
        "lambda_covar":  base_cfg["lambda_covar"],
        "lr":            base_cfg["lr"],
    })


def run_two_phase_hpo(branch, data, device, n_trials_a=100, n_trials_b=300,
                      hpo_epochs=60, hpo_iterations=3, min_k=1, max_k=20,
                      base_cfg=None, seed=42, verbose=True):
    base_cfg = base_cfg or (GLOBAL_CFG if branch == "global" else LOCAL_CFG)

    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
    sampler_a = optuna.samplers.TPESampler(seed=seed)
    study_a = optuna.create_study(direction="maximize", sampler=sampler_a, pruner=pruner,
                                  study_name=f"{branch}_phaseA")
    _enqueue_baseline_phase_a(study_a, base_cfg)
    study_a.optimize(make_objective(data, device, branch, sample_phase_a, base_cfg,
                                    hpo_epochs, hpo_iterations, min_k=min_k, max_k=max_k),
                     n_trials=n_trials_a, show_progress_bar=False, callbacks=[print_progress_callback])
    best_a = study_a.best_trial
    if verbose:
        print(f"[{branch}] Phase A best trial #{best_a.number}: {best_a.value:.4f}")
        print(f"  k_control={best_a.params['k_control']} k_pe_ghtn={best_a.params['k_pe_ghtn']} "
              f"k_gdm={best_a.params['k_gdm']}")

    phase_b_base = copy.deepcopy(base_cfg)
    phase_b_base["prototypes_per_class"] = {
        "control": best_a.params["k_control"],
        "pe_ghtn": best_a.params["k_pe_ghtn"],
        "gdm":     best_a.params["k_gdm"],
    }

    sampler_b = optuna.samplers.TPESampler(seed=seed)
    study_b = optuna.create_study(direction="maximize", sampler=sampler_b, pruner=pruner,
                                  study_name=f"{branch}_phaseB")
    _enqueue_baseline_phase_b(study_b, base_cfg)
    study_b.optimize(make_objective(data, device, branch, sample_phase_b, phase_b_base,
                                    hpo_epochs, hpo_iterations),
                     n_trials=n_trials_b, show_progress_bar=False, callbacks=[print_progress_callback])
    best_b = study_b.best_trial
    if verbose:
        print(f"[{branch}] Phase B best trial #{best_b.number}: {best_b.value:.4f}")

    final_cfg = copy.deepcopy(best_b.user_attrs["cfg"])
    final_cfg["max_epochs"], final_cfg["max_iterations"] = 200, 5
    final_cfg["run_id"] = f"{branch}_final"
    return {"branch": branch, "study_a": study_a, "study_b": study_b,
           "best_a": best_a, "best_b": best_b, "final_cfg": final_cfg}


In [0]:
import os, copy, warnings
import numpy as np
import torch, optuna
from joblibspark import register_spark
from joblib import parallel_backend, Parallel, delayed

warnings.filterwarnings("ignore", message="Spark local mode doesn't support stage-level scheduling.")
register_spark()

N_GPUS = torch.cuda.device_count()
N_WORKERS = max(N_GPUS, 1) * 4 if N_GPUS > 0 else os.cpu_count()
print(f"LOCAL branch: {N_GPUS} GPU(s) detected, running {N_WORKERS} parallel workers.")

split = build_person_split(BASE_CFG)  # SAME split as global -- same BASE_CFG, same seed, deterministic
device_load = torch.device("cpu")
local_data_full = load_local_data(LOCAL_CFG, split, device_load)


def subsample_for_hpo(data, max_train_rows=None, seed=42):
    if max_train_rows is None:
        return data
    rng = np.random.RandomState(seed)
    n_train = len(data["X_train"])
    if n_train <= max_train_rows:
        return data
    idx = rng.choice(n_train, size=max_train_rows, replace=False)
    sub = dict(data)
    for key in ["X_train", "Y_train", "pid_train", "age_train"]:
        sub[key] = data[key][idx]
    if "window_name_train" in data:
        sub["window_name_train"] = data["window_name_train"][idx]
    print(f"  HPO subsample: {n_train} -> {max_train_rows} rows "
          f"(local_data_full still holds the complete dataset)")
    return sub

# max_train_rows=None -> full local_data_full is used for HPO, no subsampling.
# Pass an explicit row count here (e.g. max_train_rows=80000) if you need to
# trade off HPO fidelity for memory/speed on a constrained cluster.
local_data = subsample_for_hpo(local_data_full, max_train_rows=80000)

STORAGE_DIR = "/local_disk0/tmp/protoehr_optuna_storage" if os.path.isdir("/local_disk0") \
              else "/tmp/protoehr_optuna_storage"
os.makedirs(STORAGE_DIR, exist_ok=True)
print(f"Optuna SQLite storage: {STORAGE_DIR}")


def make_pinned_objective_gpu(data, branch, sample_fn, base_cfg, hpo_epochs, hpo_iterations,
                              gpu_idx=None, **sample_kwargs):
    if gpu_idx is not None:
        device = torch.device(f"cuda:{gpu_idx}")
    else:
        device = torch.device("cpu")
        torch.set_num_threads(1)
    base_objective = make_objective(data, device, branch, sample_fn, base_cfg,
                                    hpo_epochs, hpo_iterations, **sample_kwargs)
    def pinned_objective(trial):
        if gpu_idx is not None:
            torch.cuda.set_device(gpu_idx)
        return base_objective(trial)
    return pinned_objective


def _optimize_chunk_gpu(study_name, storage_url, data, branch, sample_fn, base_cfg,
                        hpo_epochs, hpo_iterations, n_trials_chunk, gpu_idx, sample_kwargs):
    if n_trials_chunk <= 0:
        return
    objective = make_pinned_objective_gpu(data, branch, sample_fn, base_cfg,
                                          hpo_epochs, hpo_iterations, gpu_idx=gpu_idx, **sample_kwargs)
    study = optuna.load_study(study_name=study_name, storage=storage_url)
    study.optimize(objective, n_trials=n_trials_chunk, callbacks=[print_progress_callback])


def run_parallel_phase_gpu(study_name, storage_url, data, branch, sample_fn, base_cfg,
                           hpo_epochs, hpo_iterations, n_trials_total, n_workers, n_gpus, sample_kwargs):
    base = n_trials_total // n_workers
    rem = n_trials_total % n_workers
    chunks = [base + (1 if i < rem else 0) for i in range(n_workers)]
    gpu_assignments = [(i % n_gpus if n_gpus > 0 else None) for i in range(n_workers)]
    with parallel_backend("spark", n_jobs=n_workers):
        Parallel(n_jobs=n_workers)(
            delayed(_optimize_chunk_gpu)(study_name, storage_url, data, branch, sample_fn, base_cfg,
                                         hpo_epochs, hpo_iterations, c, g, sample_kwargs)
            for c, g in zip(chunks, gpu_assignments) if c > 0
        )


def run_two_phase_hpo_gpu(branch, data, base_cfg, n_trials_a, n_trials_b,
                          hpo_epochs, hpo_iterations, min_k, max_k, n_workers, n_gpus, seed=42):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)

    storage_url_a = f"sqlite:///{STORAGE_DIR}/{branch}_phaseA.db"
    study_a = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed),
                                  pruner=pruner, storage=storage_url_a,
                                  study_name=f"{branch}_phaseA", load_if_exists=True)
    _enqueue_baseline_phase_a(study_a, base_cfg)
    run_parallel_phase_gpu(f"{branch}_phaseA", storage_url_a, data, branch, sample_phase_a, base_cfg,
                          hpo_epochs, hpo_iterations, n_trials_a, n_workers, n_gpus,
                          {"min_k": min_k, "max_k": max_k})
    study_a = optuna.load_study(study_name=f"{branch}_phaseA", storage=storage_url_a)
    best_a = study_a.best_trial
    print(f"[{branch}] Phase A best trial #{best_a.number}: {best_a.value:.4f}  "
          f"k_control={best_a.params['k_control']} k_pe_ghtn={best_a.params['k_pe_ghtn']} "
          f"k_gdm={best_a.params['k_gdm']}")

    phase_b_base = copy.deepcopy(base_cfg)
    phase_b_base["prototypes_per_class"] = {
        "control": best_a.params["k_control"], "pe_ghtn": best_a.params["k_pe_ghtn"],
        "gdm": best_a.params["k_gdm"],
    }
    storage_url_b = f"sqlite:///{STORAGE_DIR}/{branch}_phaseB.db"
    study_b = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed),
                                  pruner=pruner, storage=storage_url_b,
                                  study_name=f"{branch}_phaseB", load_if_exists=True)
    _enqueue_baseline_phase_b(study_b, base_cfg)
    run_parallel_phase_gpu(f"{branch}_phaseB", storage_url_b, data, branch, sample_phase_b, phase_b_base,
                          hpo_epochs, hpo_iterations, n_trials_b, n_workers, n_gpus, {})
    study_b = optuna.load_study(study_name=f"{branch}_phaseB", storage=storage_url_b)
    best_b = study_b.best_trial
    print(f"[{branch}] Phase B best trial #{best_b.number}: {best_b.value:.4f}")

    final_cfg = copy.deepcopy(best_b.user_attrs["cfg"])
    final_cfg["max_epochs"], final_cfg["max_iterations"] = 200, 5
    final_cfg["run_id"] = f"{branch}_final"
    return {"branch": branch, "best_a": best_a, "best_b": best_b, "final_cfg": final_cfg}


hpo_result_local = run_two_phase_hpo_gpu(
    "local", local_data, LOCAL_CFG,
    n_trials_a=100, n_trials_b=300, hpo_epochs=30, hpo_iterations=2,
    min_k=1, max_k=20, n_workers=N_WORKERS, n_gpus=N_GPUS,
)

import json as _json
_cfg_path = os.path.join(STORAGE_DIR, "local_final_cfg.json")
with open(_cfg_path, "w") as f:
    _json.dump(hpo_result_local["final_cfg"], f, indent=2)

print("\n" + "="*70)
print(f"LOCAL FINAL CFG -- saved to {_cfg_path}")
print("(also printed below -- copy either the file or this dict into the fusion notebook)")
print("="*70)
print(hpo_result_local["final_cfg"])
